# 11 pamoka – Modelio konteksto protokolas (MCP)

**Modelio konteksto protokolas (MCP)** yra atviras standartas, leidžiantis agentams dinamiškai aptikti ir naudoti įrankius, išteklius bei duomenų šaltinius vykdymo metu. Vietoj to, kad įrankiai būtų įkoduoti tiesiogiai agentui, MCP leidžia agentams jungtis prie išorinių serverių, kurie pagal poreikį atskleidžia funkcijas.

Šioje pamokoje sužinosite:
- Kas yra MCP ir kodėl tai svarbu agentų sistemoms
- Kaip veikia MCP kliento-serverio architektūra
- Kaip kurti agentus, kurie naudoja MCP stiliaus įrankių paiešką


## Sąranka

**Išankstinės sąlygos:**
- Microsoft Foundry projektas su įdiegta modeliu
- Vykdykite `az login` autentifikacijai


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from typing import Annotated
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Kas yra Modelio Konteksto Protokolas (MCP)?

MCP apibrėžia standartinį būdą, kaip DI agentai gali atrasti ir sąveikauti su išorinėmis priemonėmis ir duomenų šaltiniais:

- **MCP serveris**: Per standartinį protokolą pateikia priemones, išteklius ir užklausas
- **MCP klientas**: Agentų vykdymo aplinka, jungiasi prie serverių ir atranda turimas galimybes
- **Dinaminis atradimas**: Agentams nereikia iš anksto sukonfigūruotų priemonių — jos atrandamos vykdymo metu

Tai galinga sistema, leidžianti kurti išplečiamus agentų sprendimus, kuriems galima pridėti naujų galimybių nekeičiant agento kodo.


## Kaip veikia MCP

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. Agentas (MCP klientas) jungiasi prie MCP serverio
2. Serveris atsako su turimų įrankių sąrašu ir jų schemomis
3. Agentas tada gali iškviesti bet kurį rastą įrankį savo mąstymo metu
4. Rezultatai grįžta tuo pačiu protokolu


## MCP įrankio aptikimo simuliacija

Kadangi tikras MCP serveris reikalauja veikiančio serverio proceso, mes pademonstruosime modelį naudodami `@tool` funkcijas, kurios imituoja, ką MCP sujungta apgyvendinimo paslauga galėtų suteikti.

Gamyboje šie įrankiai būtų dinamiškai randami iš MCP serverio, o ne apibrėžiami lokaliai.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## Agentas su MCP stiliaus įrankiais kūrimas


In [ ]:
agent = client.as_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP gamyboje

Gamybos aplinkoje MCP leidžia naudoti galingus modelius:

- **Dinaminis įrankių atradimas**: Agentai prisijungia prie MCP serverių ir vykdymo metu atranda įrankius
- **Atskira architektūra**: Įrankių tiekėjai gali atnaujinti nepriklausomai nuo agente
- **Tarporganizacinis bendrinimas**: Komandos gali per MCP serverius atskleisti galimybes, kurias gali naudoti bet kuris agentas
- **Microsoft Agent Framework palaikymas**: MAF turi integruotą MCP kliento palaikymą per `mcp` integraciją

Norint naudoti tikrą MCP serverį su MAF, jungiamasi per `hosted_mcp_tool()` arba MCP kliento integraciją.

**Sužinokite daugiau:**
- [MCP specifikacija](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP palaikymas](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Summary

In this lesson, you learned:
- **MCP** is an open standard for dynamic tool discovery between agents and tool providers
- The **client-server architecture** lets agents discover capabilities at runtime
- MCP enables **extensible, decoupled agent systems** where tools can be added without code changes
- Microsoft Agent Framework provides **built-in MCP support** for production use


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, prašome atkreipti dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba laikomas autoritetingu šaltiniu. Svarbiai informacijai rekomenduojama naudoti profesionalų žmogiškąjį vertimą. Mes neatsakome už jokius nesusipratimus ar neteisingą interpretaciją, kilusią naudojantis šiuo vertimu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
